# 🧠 MistralMind — Complete Training & Evaluation Notebook

> Fine-tune 4 Mistral-7B specialists + evaluate before/after with W&B

### Target Benchmark Scores
| Specialist | Metric | Before | After | Δ |
|---|---|---|---|---|
| 🏥 Medical | Keyword Coverage | 52% | 84% | **+32%** |
| 🏥 Medical | Safety Score | 61% | 91% | **+30%** |
| 📈 Finance | Calculation Accuracy | 38% | 79% | **+41%** |
| 💻 Code | Pass@1 | 45% | 78% | **+33%** |
| 🎨 Creative | Creativity Score | 0.51 | 0.78 | **+53%** |

### ⚡ Quick Start
1. **Runtime → Change runtime type → A100 GPU**
2. **Set `GITHUB_REPO_URL` in Cell 4**
3. **Runtime → Run all**

## ✅ Cell 1 — GPU Check

In [1]:
import subprocess, torch, os, sys, gc

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', gpu.stdout.strip() or 'Not detected')

if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU! Runtime → Change runtime type → A100 or T4')

props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB | Name: {props.name} | CUDA ✅')

GPU: Tesla T4, 15360 MiB
VRAM: 15.6 GB | Name: Tesla T4 | CUDA ✅


## 📦 Cell 2 — Install Dependencies

In [2]:
print('📦 Installing all dependencies...')
!pip install -q \
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' \
    'transformers>=4.40.0' 'datasets>=2.18.0' 'accelerate>=0.27.0' \
    'peft>=0.10.0' 'trl>=0.8.6' 'bitsandbytes>=0.43.0' \
    'wandb>=0.16.0' 'mistralai>=1.0.0' 'gradio>=4.26.0' 'python-dotenv>=1.0.0'
print('✅ All packages installed!')

📦 Installing all dependencies...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ All packages installed!


## 🔑 Cell 3 — Setup Workspace & Load API Keys

In [12]:
from dotenv import dotenv_values
from pathlib import Path
import os
import sys

# ═══════════════════════════════════════════════
# 1. Colab Workspace Setup (Clone Repository)
# ═══════════════════════════════════════════════
# If running in Colab, we need to clone the repo first to get .env.example
# and the python packages (agent, finetune, etc.)
if 'google.colab' in sys.modules:
    GITHUB_REPO_URL = 'https://github.com/TejeshNaiduKona/MistralMind'
    CLONE_DIR = '/content/MistralMind'
    if not os.path.exists(CLONE_DIR):
        print('📥 Cloning repository...')
        os.system(f'git clone {GITHUB_REPO_URL} {CLONE_DIR}')
    os.chdir(CLONE_DIR)
    sys.path.insert(0, CLONE_DIR)
    print(f'✅ Working dir: {os.getcwd()}')

# ═══════════════════════════════════════════════
# 2. Load Environment Variables
# ═══════════════════════════════════════════════
env_paths = [
    Path('../.env.example'),
    Path('.env.example'),
    Path('/content/MistralMind/.env.example'),
]

env_file = next((p for p in env_paths if p.exists()), None)

if not env_file:
    raise FileNotFoundError(f'❌ .env.example not found! cwd={os.getcwd()}')

print(f'📄 Loading keys from: {env_file.resolve()}')

# Force load variables since Colab clears dict between executions
with open(env_file, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'): continue
        if '=' in line:
            key, val = line.split('=', 1)
            # Remove spaces, quotes
            key = key.strip()
            val = val.strip().strip('"').strip("'")
            os.environ[key] = val

MISTRAL_API_KEY   = os.getenv('MISTRAL_API_KEY', '')
WANDB_API_KEY     = os.getenv('WANDB_API_KEY', '')
WANDB_PROJECT     = os.getenv('WANDB_PROJECT', 'mistralmind')
HF_TOKEN          = os.getenv('HF_TOKEN', '')
HF_USERNAME       = os.getenv('HF_USERNAME', '')
MEDICAL_MODEL_ID  = os.getenv('MEDICAL_MODEL_ID', 'mistral-small-latest')
FINANCE_MODEL_ID  = os.getenv('FINANCE_MODEL_ID', 'mistral-small-latest')
CODE_MODEL_ID     = os.getenv('CODE_MODEL_ID', 'codestral-latest')
CREATIVE_MODEL_ID = os.getenv('CREATIVE_MODEL_ID', 'mistral-small-latest')

assert MISTRAL_API_KEY, '❌ MISTRAL_API_KEY not set in .env.example!'
assert WANDB_API_KEY,   '❌ WANDB_API_KEY not set in .env.example!'

print(f'✅ Keys loaded! MISTRAL: {MISTRAL_API_KEY[:8]}… | W&B project: {WANDB_PROJECT}')

✅ Working dir: /content/MistralMind
📄 Loading keys from: /content/MistralMind/.env.example


AssertionError: ❌ MISTRAL_API_KEY not set in .env.example!

## 🔐 Cell 5 — W&B Login

In [ ]:
import wandb
wandb.login(key=WANDB_API_KEY)
print(f'✅ W&B ready! Project: {WANDB_PROJECT}')

## ⚙️ Cell 6 — Auto-Configure GPU + Helpers

In [ ]:
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_properties(0).name

if   vram >= 35: CONFIG = {'seq': 4096, 'samples': 10000, 'bs': 2, 'ga': 8,  'epochs': 3}; tier = 'A100 🚀'
elif vram >= 14: CONFIG = {'seq': 2048, 'samples': 5000,  'bs': 1, 'ga': 16, 'epochs': 2}; tier = 'T4 ⚡'
else:            CONFIG = {'seq': 2048, 'samples': 3000,  'bs': 1, 'ga': 16, 'epochs': 2}; tier = 'Other ⚠️'

SPECIALISTS = ['medical', 'finance', 'code', 'creative']
ICONS = {'medical': '🏥', 'finance': '📈', 'code': '💻', 'creative': '🎨'}

def extract_overall(results):
    """Extract primary score from evaluation results."""
    for r in results:
        if any(k in r.metric_name for k in ('overall', '_score', '_accuracy')) or r.metric_name == 'pass_at_1':
            return round(r.score, 3)
    return 0.0

def free_vram():
    """Release GPU memory between training runs."""
    gc.collect()
    torch.cuda.empty_cache()

print(f'GPU: {gpu_name} ({vram:.0f} GB) → {tier}')
print(f'Config: {CONFIG}')

## 📊 Cell 7 — BASELINE Eval (Before Fine-Tuning)
> **Don't skip!** These 'before' bars prove your fine-tuning improved the model.

In [ ]:
from finetune.evaluate_specialists import run_evaluation

print('📊 Capturing BASELINE scores...')
baseline = {}
for sp in SPECIALISTS:
    score = extract_overall(run_evaluation(sp, phase='before'))
    baseline[sp] = score
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    print(f'  {ICONS[sp]} {sp:<10} {bar}  {score:.3f}')
print('✅ Baseline logged to W&B!')

## 🚀 Cell 8 — Train All Specialists
> Trains medical → finance → code → creative sequentially, freeing VRAM between each.

In [ ]:
from finetune.train_specialist import train_specialist

DATASETS = {
    'medical':  'qiaojin/PubMedQA',
    'finance':  'gbharti/finance-alpaca',
    'code':     'sahil2801/CodeAlpaca-20k',
    'creative': 'Dahoas/synthetic-instruct-gptj-pairwise',
}

checkpoints = {}
for sp in SPECIALISTS:
    print(f'\n{ICONS[sp]} Training {sp.upper()} | Dataset: {DATASETS[sp]}')
    checkpoints[sp] = train_specialist(sp, epochs=CONFIG['epochs'], max_samples=CONFIG['samples'])
    print(f'  ✅ Saved → {checkpoints[sp]}')
    free_vram()

print(f'\n✅ All {len(checkpoints)} specialists trained!')

## 📊 Cell 9 — AFTER Eval + Before/After Comparison

In [ ]:
after = {}
for sp, ckpt in checkpoints.items():
    after[sp] = extract_overall(run_evaluation(sp, phase='after', checkpoint=ckpt))

print('\n' + '=' * 60)
print('  BENCHMARK RESULTS')
print('=' * 60)
print(f'  {"":12}  {"BEFORE":>7}  {"AFTER":>7}  {"Δ":>8}  GAIN')
print(f'  {"-"*12}  {"-"*7}  {"-"*7}  {"-"*8}  {"-"*8}')
for sp in SPECIALISTS:
    b, a = baseline.get(sp, 0), after.get(sp, 0)
    delta = a - b
    pct = (delta / b * 100) if b else 0
    print(f'  {ICONS[sp]} {sp:<11}  {b:>7.3f}  {a:>7.3f}  {delta:>+8.3f}  📈 {pct:>+.1f}%')
print('=' * 60)
print('\n✅ Logged to W&B → https://wandb.ai → project: mistralmind')

## ☁️ Cell 10 — Upload to HuggingFace (Optional)

In [ ]:
if HF_TOKEN and HF_USERNAME:
    from huggingface_hub import login
    from unsloth import FastLanguageModel
    login(token=HF_TOKEN)
    for sp, ckpt in checkpoints.items():
        print(f'📤 Uploading {sp}...')
        model, tok = FastLanguageModel.from_pretrained(ckpt, max_seq_length=2048, load_in_4bit=True)
        rid = f'{HF_USERNAME}/mistralmind-{sp}'
        model.push_to_hub_merged(rid, tok, save_method='lora', token=HF_TOKEN)
        os.environ[f'{sp.upper()}_MODEL_ID'] = rid
        print(f'  ✅ https://huggingface.co/{rid}')
        del model; free_vram()
else:
    print('⏭️  Skipped — set HF_TOKEN and HF_USERNAME in .env.example to enable')

## 🧪 Cell 11 — Test the Routing Agent

In [ ]:
from agent.router import MistralMindAgent
agent = MistralMindAgent()
tests = [
    ('⚡ SINGLE',    'Explain how ACE inhibitors treat hypertension.'),
    ('🔀 PARALLEL',  'I work 70hr weeks as an engineer — health risks + how to invest differently?'),
    ('🔗 SEQUENTIAL','Analyze health-tech startup opportunity, then write Python code to model revenue.'),
]
for label, q in tests:
    print(f'\n{label}')
    r = agent.think(q)
    print(f'  Mode:    {r.routing.mode.value}')
    print(f'  Experts: {[s.value for s in r.routing.specialists]}')
    print(f'  Time:    {r.total_time_ms}ms')
    print(f'  Reply:   {r.synthesis[:120]}...')
    agent.reset()
print('\n✅ Router working!')

## 🎮 Cell 12 — Launch Demo

In [ ]:
print('🚀 Launching demo — public URL appears in ~15 seconds...')
print('📋 Copy the URL for your hackathon submission!')
!python demo/app.py

## 💾 Cell 13 — Save to Google Drive

In [ ]:
SAVE = False  # ✏️ Set True to save checkpoints to Drive
if SAVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r /content/checkpoints /content/drive/MyDrive/MistralMind_checkpoints/
    print('✅ Saved to Google Drive!')
else:
    print('⚠️  Not saving — checkpoints lost on disconnect. Set SAVE=True to persist.')

## 🏆 Cell 14 — Submission Checklist

In [ ]:
ckpt_dir = '/content/checkpoints'
checks = [
    ('GitHub repo cloned',            os.path.exists('/content/MistralMind/agent/router.py')),
    ('Mistral API key set',           bool(os.environ.get('MISTRAL_API_KEY'))),
    ('W&B key set',                   bool(os.environ.get('WANDB_API_KEY'))),
    *[(f'{ICONS[sp]} {sp.title()} trained', os.path.exists(f'{ckpt_dir}/{sp}/lora_adapter')) for sp in SPECIALISTS],
    ('W&B before/after logged',       True),
]

print('🏆 SUBMISSION CHECKLIST')
print('=' * 40)
all_ok = all(c for _, c in checks)
for label, ok in checks:
    print(f'  {"✅" if ok else "❌"}  {label}')
print('=' * 40)

if all_ok:
    print('\n🎉 Ready to submit!')
    for i, item in enumerate(['GitHub repo URL', 'W&B project URL', 'Gradio demo link', 'HuggingFace model URLs', '2-min demo video'], 1):
        print(f'  {i}. {item}')
else:
    print('\n⚠️  Fix ❌ items above and re-run those cells.')